# سبق 02 - مائیکروسافٹ ایجنٹ فریم ورک کی تحقیق

**Microsoft Agent Framework (MAF)** ایک یکساں فریم ورک ہے جو AI ایجنٹس بنانے کے لیے ہے۔ یہ ایک صاف، قابل ترتیب معمار فراہم کرتا ہے جس میں چار بنیادی اجزاء شامل ہیں:

- **کلائنٹ** – AI ماڈل اینڈپوائنٹ سے جڑتا ہے اور مواصلات کو سنبھالتا ہے
- **ایجنٹ** – کلائنٹ کو ہدایات اور ٹول کی وضاحتوں کے ساتھ لپیٹتا ہے
- **ٹولز** – ایجنٹ کی صلاحیتوں کو ذاتی فنکشنز کے ساتھ بڑھاتے ہیں جنہیں ماڈل کال کر سکتا ہے
- **سیشن** – کثیر مرحلہ بات چیت کے لیے گفتگو کی تاریخ کو برقرار رکھتا ہے

اس سبق میں، ہم ایک **ٹریول بکنگ ایجنٹ** بنائیں گے جو ان تصورات کو استعمال کرتے ہوئے منزل کی دستیابی کی جانچ کرے گا۔


## ترتیب


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q
! pip install python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

## ایجنٹ فریم ورک کے فن تعمیر کو سمجھنا

مائیکروسافٹ ایجنٹ فریم ورک ایک تہہ دار فن تعمیر کی پیروی کرتا ہے:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **کلائنٹ** – ایک `FoundryChatClient` Azure OpenAI ڈپلائمنٹ سے جڑتا ہے۔ یہ توثیق، درخواست کی تشکیل، اور جواب کی تشریح کو سنبھالتا ہے۔
2. **ایجنٹ** – کلائنٹ سے `provider.create_agent()` کے ذریعے بنایا جاتا ہے، ایجنٹ ماڈل کی رسائی کو ہدایات (سسٹم پرامپٹ) اور آلات کے ساتھ جوڑتا ہے۔
3. **آلات** – پائتھن فنکشنز جن پر `@tool` کی سجاوٹ کی گئی ہے جنہیں ایجنٹ عمل انجام دینے یا ڈیٹا حاصل کرنے کے لیے کال کر سکتا ہے۔
4. **سیشن** – ایک `AgentSession` آبجیکٹ (جو `agent.create_session()` کے ذریعے بنایا جاتا ہے) جو گفتگو کی تاریخ کو محفوظ کرتا ہے، اس طرح کثیر مرحلے کی بات چیت کو قابل بناتا ہے جہاں ایجنٹ پچھلا تناظر یاد رکھتا ہے۔

آئیے ہر تہہ کو مرحلہ وار بنائیں۔


In [ ]:
# Create the client – this is the connection to the AI model
endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## @tool سجاوٹ کے ساتھ ٹولز شامل کرنا

ٹولز ایجنٹس کو ٹیکسٹ بنانے سے آگے اقدامات کرنے دیتے ہیں۔ `@tool` سجاوٹ ایک عام پائتھون فنکشن کو اس چیز میں تبدیل کرتی ہے جسے ایجنٹ کال کر سکتا ہے۔

اہم نکات:
- ماڈل کو ہر پیرامیٹر سمجھنے کے لیے `Annotated[type, "description"]` کا استعمال کریں۔
- ڈاکسٹرنگ ٹول کی وضاحت بن جاتی ہے جو ماڈل دیکھتا ہے۔
- `approval_mode="never_require"` کا مطلب ہے کہ ٹول بغیر صارف کی تصدیق کے خودکار طور پر چلتا ہے۔


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## ٹولز کے ساتھ ایک ایجنٹ تخلیق کرنا

اب ہم کلائنٹ، ہدایات، اور ٹولز کو ایک ایجنٹ میں جوڑتے ہیں۔ `instructions` نظام کے اشارے کے طور پر کام کرتی ہیں — یہ ایجنٹ کی شخصیت اور رویے کی تعریف کرتی ہیں۔


In [ ]:
agent = provider.as_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## سیشنز کے ساتھ کثیر دور کی بات چیت  

ایک `AgentSession` (جو `agent.create_session()` کے ذریعے بنایا جاتا ہے) گفتگو میں تمام پیغامات کا ریکارڈ رکھتا ہے۔ ایک ہی سیشن کو ہر `agent.run()` کال میں فراہم کرکے، ایجنٹ کو پوری گفتگو کی تاریخ تک رسائی حاصل ہوتی ہے اور وہ پہلے کے پیغامات کی طرف واپس رجوع کر سکتا ہے۔  

ہم `tools=[check_destination_availability]` پاس کرتے ہیں تاکہ ایجنٹ ہر دور میں ہمارا دستیابی چیک کرنے والا کال کر سکے۔  


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## خلاصہ

اس سبق میں آپ نے مائیکروسافٹ ایجنٹ فریم ورک کے چار ستونوں کا جائزہ لیا:

| تصور | آپ نے کیا سیکھا |
|---------|------------------|
| **کلائنٹ** | `FoundryChatClient` اسناد پر مبنی توثیق کے ساتھ Azure OpenAI سے جڑتا ہے |
| **ایجنٹ** | `provider.create_agent()` ماڈل کنکشن کو ہدایات اور نام کے ساتھ جوڑتا ہے |
| **ٹولز** | `@tool` ڈیکوریٹر پائتھن فنکشنز کو ایجنٹ کی کال کے لیے ظاہر کرتا ہے |
| **سیشن** | `agent.create_session()` متعدد موڑوں کے دوران گفتگو کی تاریخ کو برقرار رکھتا ہے |

یہ تعمیراتی بلاکس مل کر ایسے ایجنٹس تخلیق کرتے ہیں جو قدرتی گفتگو کر سکتے ہیں، بیرونی فنکشنز کو کال کر سکتے ہیں، اور سیاق و سباق کو برقرار رکھ سکتے ہیں — جو کہ بعد کے اسباق میں مزید ترقی یافتہ ایجنٹک نمونوں کی بنیاد ہے۔


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ڈس کلیمر**:
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے ترجمہ کی گئی ہے۔ جبکہ ہم درستگی کے لیے کوشاں ہیں، براہ کرم اس بات سے آگاہ رہیں کہ خودکار ترجمے میں غلطیاں یا عدم درستیاں ہو سکتی ہیں۔ اصل دستاویز اپنے مادری زبان میں مستند ماخذ سمجھی جائے گی۔ حساس معلومات کے لیے پیشہ ور انسانی ترجمہ کی سفارش کی جاتی ہے۔ اس ترجمے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تشریح کی ذمہ داری ہم قبول نہیں کرتے۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
